# 05 - Machine Learning, MLflow y Model Registry

Este notebook construye un dataset Gold para Machine Learning, entrena modelos de clasificación, registra métricas en MLflow y registra el mejor modelo en Unity Catalog.

## Problema

Clasificar si un accidente presenta víctimas:

- `tiene_victimas = 0`: solo daños.
- `tiene_victimas = 1`: con heridos o muertos.

El objetivo es una clasificación al momento conceptual del registro del accidente, no una predicción previa a que el accidente ocurra.

## Prevención de data leakage

No se utilizan como features `gravedad`, `nro_heridos`, `numero_muertos` ni información de la tabla de víctimas, porque revelan directa o indirectamente la variable objetivo.


## 1. Configuración


In [ ]:
from pyspark.sql import functions as F

dbutils.widgets.text("catalog_name", "accidentalidad_databricks", "Catalog del proyecto")

PROY_CATALOG = dbutils.widgets.get("catalog_name")
PROY_SCHEMA_SILVER = "silver"
PROY_SCHEMA_GOLD = "gold"

SOURCE_TABLE = f"{PROY_CATALOG}.{PROY_SCHEMA_SILVER}.accidentes"
FEATURE_TABLE = f"{PROY_CATALOG}.{PROY_SCHEMA_GOLD}.features_accidentalidad"
MODEL_NAME = f"{PROY_CATALOG}.{PROY_SCHEMA_GOLD}.modelo_riesgo_victimas"

print(f"Fuente:   {SOURCE_TABLE}")
print(f"Features: {FEATURE_TABLE}")
print(f"Modelo:   {MODEL_NAME}")

## 2. Construcción de `gold.features_accidentalidad`

Se mantiene una fila por accidente (`croquis`). Las variables se construyen únicamente con información contextual disponible en el registro del accidente.


In [ ]:
df_accidentes = spark.table(SOURCE_TABLE)

df_features = (
    df_accidentes
    .select(
        "croquis",
        "anio_accidente",
        "mes_accidente",
        F.dayofweek("fecha_accidente").alias("dia_semana"),
        F.regexp_extract("hora_accidente", r"^(\d{1,2})", 1).cast("int").alias("hora"),
        "localidad",
        "clase_accidente",
        "coordenada_x",
        "coordenada_y",
        F.when(F.col("gravedad") == "SOLO DAÑOS", F.lit(0))
         .when(F.col("gravedad").isin("CON HERIDOS", "CON MUERTOS"), F.lit(1))
         .otherwise(F.lit(None))
         .cast("int")
         .alias("tiene_victimas")
    )
    .filter(F.col("tiene_victimas").isNotNull())
)

(
    df_features.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(FEATURE_TABLE)
)

print(f"[OK] Tabla creada: {FEATURE_TABLE}")
print(f"Registros: {df_features.count():,}")
display(df_features.limit(10))

## 3. Validación de la variable objetivo


In [ ]:
df_target = (
    spark.table(FEATURE_TABLE)
    .groupBy("tiene_victimas")
    .count()
)

total_registros = spark.table(FEATURE_TABLE).count()

df_target = (
    df_target
    .withColumn(
        "porcentaje",
        F.round(F.col("count") / F.lit(total_registros) * 100, 2)
    )
    .orderBy("tiene_victimas")
)

display(df_target)

## 4. Preparación del dataset de entrenamiento

La evaluación utiliza un corte temporal:

- entrenamiento: 2018-2021;
- prueba: 2022.

Esto es más realista que un split aleatorio porque evalúa el modelo sobre un periodo posterior al usado para entrenar.


In [ ]:
df_ml = spark.table(FEATURE_TABLE).toPandas()

feature_columns = [
    "mes_accidente",
    "dia_semana",
    "hora",
    "localidad",
    "clase_accidente",
    "coordenada_x",
    "coordenada_y",
]

train_df = df_ml[df_ml["anio_accidente"] < 2022].copy()
test_df = df_ml[df_ml["anio_accidente"] == 2022].copy()

X_train = train_df[feature_columns]
y_train = train_df["tiene_victimas"].astype(int)

X_test = test_df[feature_columns]
y_test = test_df["tiene_victimas"].astype(int)

print(f"Train: {len(X_train):,} registros")
print(f"Test:  {len(X_test):,} registros")
print(f"Tasa positiva train: {y_train.mean():.4f}")
print(f"Tasa positiva test:  {y_test.mean():.4f}")

## 5. Pipeline de preprocesamiento


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

numeric_features = [
    "mes_accidente",
    "dia_semana",
    "hora",
    "coordenada_x",
    "coordenada_y",
]

categorical_features = [
    "localidad",
    "clase_accidente",
]

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ]
)

## 6. MLflow y entrenamiento de modelos

Se entrenan dos candidatos y se elige el mejor por F1, una métrica adecuada cuando existe desbalance entre clases.


In [ ]:
import inspect
import pandas as pd
import mlflow
import mlflow.sklearn

from mlflow.models import infer_signature
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
)

current_user = spark.sql("SELECT current_user() AS user").first()["user"]
EXPERIMENT_NAME = f"/Users/{current_user}/accidentalidad_databricks_ml"

mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(EXPERIMENT_NAME)

models = {
    "logistic_regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42,
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    ),
}

results = []

for model_name, estimator in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", estimator),
    ])

    with mlflow.start_run(run_name=model_name) as run:
        pipeline.fit(X_train, y_train)

        y_pred = pipeline.predict(X_test)
        y_prob = pipeline.predict_proba(X_test)[:, 1]

        metrics = {
            "accuracy": accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "f1": f1_score(y_test, y_pred, zero_division=0),
            "roc_auc": roc_auc_score(y_test, y_prob),
        }

        mlflow.log_param("model_name", model_name)
        mlflow.log_param("train_period", "2018-2021")
        mlflow.log_param("test_period", "2022")
        mlflow.log_param("feature_count", len(feature_columns))
        mlflow.log_metrics(metrics)
        mlflow.log_dict(
            classification_report(y_test, y_pred, output_dict=True, zero_division=0),
            "classification_report.json",
        )

        signature = infer_signature(X_train, pipeline.predict(X_train))
        input_example = X_train.head(5)

        log_model_kwargs = {
            "sk_model": pipeline,
            "signature": signature,
            "input_example": input_example,
        }

        if "name" in inspect.signature(mlflow.sklearn.log_model).parameters:
            log_model_kwargs["name"] = "model"
        else:
            log_model_kwargs["artifact_path"] = "model"

        model_info = mlflow.sklearn.log_model(**log_model_kwargs)

        results.append({
            "model": model_name,
            "run_id": run.info.run_id,
            "model_uri": model_info.model_uri,
            **metrics,
        })

results_df = pd.DataFrame(results).sort_values("f1", ascending=False)
display(results_df)

## 7. Selección y registro del mejor modelo en Unity Catalog


In [ ]:
best = results_df.iloc[0]

print(f"Mejor modelo: {best['model']}")
print(f"F1:           {best['f1']:.4f}")
print(f"ROC-AUC:      {best['roc_auc']:.4f}")
print(f"Model URI:    {best['model_uri']}")

registered_model = mlflow.register_model(
    model_uri=best["model_uri"],
    name=MODEL_NAME,
)

print(f"[OK] Modelo registrado: {MODEL_NAME}")
print(f"Versión: {registered_model.version}")

## 8. Alias `Champion`

El alias desacopla los consumidores del número de versión. Una futura versión puede convertirse en Champion sin cambiar el código de inferencia.


In [ ]:
from mlflow import MlflowClient

client = MlflowClient(registry_uri="databricks-uc")

client.set_registered_model_alias(
    name=MODEL_NAME,
    alias="Champion",
    version=registered_model.version,
)

champion = client.get_model_version_by_alias(MODEL_NAME, "Champion")

print(f"[OK] Alias Champion -> versión {champion.version}")

## 9. Validación de inferencia desde Model Registry


In [ ]:
champion_uri = f"models:/{MODEL_NAME}@Champion"
champion_model = mlflow.pyfunc.load_model(champion_uri)

sample = X_test.head(5).copy()
sample["prediccion_tiene_victimas"] = champion_model.predict(sample)

display(sample)

## 10. Resumen

Con este notebook se demuestra:

- construcción de una tabla Gold específica para ML;
- definición explícita de la variable objetivo;
- prevención de data leakage;
- evaluación temporal 2018-2021 vs. 2022;
- comparación de dos modelos;
- tracking de parámetros, métricas y artefactos con MLflow;
- registro del mejor modelo en Unity Catalog;
- asignación del alias `Champion`;
- carga del modelo desde Model Registry para inferencia.
